# 第7回 総合演習：データを選び、しきい値を誤りの重さで決める（解答例・教員用）


---


## 前回からの接続


第6回は、携帯電話のメッセージから迷惑メッセージを見分け、混同行列（confusion matrix）で評価しました。ロジスティック回帰（logistic regression）で確率を出すと、しきい値を動かすたびに、見逃しと誤検出が入れ替わりました。

今日は総合演習です。データを選ぶところから、1本の解析を通します。期末レポートでやることを、ひと通り先にやってみます。そして、第6回で残った「どこで切ればよいか」に、2つの誤りの重さから答えます。


---


## 今日の分析目標


ネットショップの訪問から、購入を見分けられるか。見分けたとして、どこで切ればよいかに答える。

まず、公開データの置き場（UCI機械学習リポジトリ）で候補を比べ、データを選びます。次に、12本の列からロジスティック回帰で購入の確率を出し、混同行列とAUCで測ります。その成績を信じてよいかを、列を1本ずつ外して確かめます。最後に、見逃しと誤検出に重さを置き、損の合計がいちばん小さいしきい値を選びます。

もう1つのねらいは、解析の手順を関数にまとめ、入力を替えて何度も呼べるようになることです。「列を選ぶ → 交差検証（cross-validation）で確率 → AUC」を1つの関数にし、列の組を替えて呼びます。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

世の中には、誰でも使える公開データが山ほどあります。第6回のSMSも、UCI機械学習リポジトリという置き場のデータでした。今日は、その置き場から、データを選ぶところを見せます。期末レポートでは、データを選ぶところから自分で行います。


### 場面

ネットショップには、毎日たくさんの人が訪れます。ページを見て回り、買う人もいれば、何も買わずに帰る人もいます。

購入しそうな訪問に、チャットの担当者が声をかけたい。ただし、担当者の数は限られるので、誰にでも声をかけるわけにはいきません。どの訪問に声をかけるかを、購入の確率で決めたいのです。

誤りは2種類あります。買う気のある訪問に声をかけそびれる誤り（見逃し）と、買わない訪問に担当者の時間を使う誤り（誤検出）です。2つの誤りの重さは、同じではありません。


### 手元のデータ

Online Shoppers Purchasing Intentionは、あるネットショップの1年間の訪問を集めたデータです。1行が1回の訪問（セッション）で、12,330件あります。見たページの種類ごとの数と時間、直帰率・離脱率、ページ価値などの列と、購入したかどうかの列（Revenue）があります。

データはUCI機械学習リポジトリで公開されていて（CC BY 4.0）、出典（Sakar, Polat, Katircioglu, Kastro, 2019）を書けば使えます。


### 最後に出す答え

訪問の様子から、購入をどのくらい見分けられるか。そして、確率をどこで切って、声をかける訪問を決めるかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. データを選ぶ：UCIで3本の候補を比べ、実際の記録のネットショップを選びます。購入は0.155で、何も購入と言わなくても正解率は0.845です。
2. 作って測る：12本の列から、ロジスティック回帰で購入の確率を出します。AUCは0.877です。0.5で切ると、買った訪問の6割を見逃します。
3. 確かめる：えっ、列を1本ずつ外すと、ページ価値を外したときだけ、AUCが0.718まで下がります。データの説明を読むと、ページ価値は購入の売上から計算された値でした。使ってよいかは、いつの購入から計算したかで決まります。
4. しきい値を決める：見逃しを誤検出の5倍と置くと、0.15で切るのが、損の合計がいちばん小さくなります。

このあと、この4つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| 選ぶ・予想（1・2節） | どのデータで、何を予想する？ | UCIで候補を比べて選び、読み込み、予想を書く |
| 作って測る（3節） | 購入を見分けられる？ | 12本の列で確率を出し、混同行列とAUCで測る |
| 確かめる（4節） | その成績、信じてよい？ | 手順を関数にし、列を1本ずつ外す。データの説明を読む |
| しきい値を決める（5節） | どこで切る？ | 2つの誤りに重さを置き、損の合計がいちばん小さいところで切る |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 公開データのページを読み、1行・答えの列・件数・出どころ（実際の記録か、作ったデータか）を確かめて、データを選べる
- 標準化（standardization）とロジスティック回帰をつなぎ、交差検証で確率を出して、混同行列とAUCで測れる
- 解析の手順を関数にまとめ、列の組を替えて呼び、どの列が効いているかを確かめられる
- 成績を決めている列が何から作られたかを、データの説明で確かめられる
- 見逃しと誤検出に重さを置き、損の合計がいちばん小さいしきい値を選べる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出し、UCIのデータを取ってくる道具`ucimlrepo`を入れる
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
!pip install -q ucimlrepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


---


## 1. どのデータを使う？


ここから、段1「選ぶ・予想」（1・2節）です。問いは、どのデータで、何を予想するか、です。

まず、公開データの置き場から、データを選びます。


### UCI機械学習リポジトリ

UCI機械学習リポジトリ（https://archive.ics.uci.edu/）は、1987年から、研究に使われたデータを集めて公開している置き場です。1つのデータに1つのページがあり、データの番号が付いています。第6回のSMSも、ここのデータでした。表だけでなく、文章や画像のデータもあります。

「分類（classification）」「表の形」「件数」などで、候補を絞り込めます。候補ごとに、データのページを開いて読みます。名前だけで決めず、ページの説明まで読みます。


### ページで見ること

| 見ること | 中身 |
|---|---|
| 中身 | 1行は何か（1人・1回・1つ）。答えの列はどれか、2択か量か。件数と答えの割合 |
| 出どころ | ライセンスと引用する文献。実際の記録か、作ったデータ（合成データ）か。列ごとの説明 |
| 扱い | 欠測はあるか。人を判定するデータ（審査・診断など）は、扱いに配慮が要る |

授業や期末レポートでは、身近で、人を判定しないデータが扱いやすいです。


### 候補を3本、表で比べる

| 候補 | 1行 | 件数 | 答えの割合 | 出どころ |
|---|---|---:|---:|---|
| ネットショップ（番号468） | 訪問1回 | 12,330 | 購入15.5% | 実際の訪問 |
| 工作機械（番号601） | 加工1回 | 10,000 | 故障3.4% | 作ったデータ |
| きのこ（番号848） | きのこ1本 | 61,069 | 毒55.5% | 作ったデータ |

どれも、2択を当てる分類のデータで、ライセンスはCC BY 4.0です。

工作機械のページには「synthetic dataset」（合成データ）とあります。本物の工場の記録は手に入りにくいので、それらしく作った、と書いてあります。作ったデータからの発見は、作った人の決めごとの発見かもしれません。きのこのページには「61069 hypothetical mushrooms」（架空のきのこ）とあり、図鑑のきのこの種をもとに作ったデータです。列20本のうち、数の列は3本だけです。

ページの件数と答えの割合は、取ってきて確かめられます（工作機械は故障の列、きのこは毒（p）の列）。


In [ ]:
# @title 候補2本の件数と答えの割合（中身は読み飛ばしてよい）
from ucimlrepo import fetch_ucirepo
for k in [601, 848]:
    d = fetch_ucirepo(id=k)
    y = d.data.targets.iloc[:, 0]
    print(k, len(y), '件', y.value_counts(normalize=True).round(3).to_dict())


そこで、ネットショップに決めます。実際のネットショップの1年間の訪問の記録で、1行が身近です。購入は15.5%で、少なすぎず、偏りもあります。数の列が10本あり、今日の道具でそのまま扱えます。


### お手本：UCIから読み込む

- 役割：データの番号468で、ネットショップの表をUCIから取ってくる
- 入力：データの番号`468`
- 出力：表`raw`（12,330行、18列）


In [ ]:
from ucimlrepo import fetch_ucirepo
uci = fetch_ucirepo(id=468)
raw = pd.concat([uci.data.features, uci.data.targets], axis=1)
print(raw.shape)


#### 1行ずつ読む

- 1行目：UCIのデータを取ってくる道具`fetch_ucirepo`を呼び出す
- 2行目：データの番号468を渡して取ってきて、`uci`にしまう
- 3行目：特徴の表と答えの表を、横につなぐ（`pd.concat`、`axis=1`は横）
- 4行目：表の形（行の数、列の数）を表示する


#### fetch_ucirepoがしていること

データのページに置いてある表を、ネットから取ってきます。特徴の列（`uci.data.features`）と、答えの列（`uci.data.targets`）に分けて渡します。ページの説明（`uci.metadata`）も、一緒に届きます。


#### 中身を見る：答えの表


In [ ]:
uci.data.targets.head(3)


答えの列はRevenue（売上）の1本。購入したらTrue、しなければFalse


### お手本：列の名前を日本語に

- 役割：英語の列の名前を、日本語の名前に替える
- 入力：表`raw`と、名前の組の辞書`names`
- 出力：表`df`

元の名前を鍵に、日本語の名前を値にした辞書です（第5回）。


In [ ]:
names = {'Administrative': '管理ページ数',
         'Administrative_Duration': '管理ページ時間',
         'Informational': '情報ページ数',
         'Informational_Duration': '情報ページ時間',
         'ProductRelated': '商品ページ数',
         'ProductRelated_Duration': '商品ページ時間',
         'BounceRates': '直帰率',
         'ExitRates': '離脱率',
         'PageValues': 'ページ価値',
         'SpecialDay': '特別な日'}
df = raw.rename(columns=names)


#### 1行ずつ読む

- 1〜10行目：元の列の名前と、日本語の名前の組を10個、辞書`names`にしまう
- 11行目：`rename(columns=names)`で、辞書のとおりに列の名前を替え、`df`にしまう


#### 中身を見る：辞書で引く


In [ ]:
print(names['PageValues'])
print(len(names))


鍵（元の名前）を渡すと、値（日本語の名前）が返る。組は10個


### お手本：0と1の列と、使う列

- 役割：週末・新規・購入を0と1の列にし、手がかりに使う12本の列の名前をリストにする
- 入力：表`df`
- 出力：列「週末」「新規」「購入」と、リスト`cols`


In [ ]:
df['週末'] = df['Weekend'].astype(int)
df['新規'] = (df['VisitorType'] == 'New_Visitor').astype(int)
df['購入'] = df['Revenue'].astype(int)
cols = ['管理ページ数', '管理ページ時間', '情報ページ数',
        '情報ページ時間', '商品ページ数', '商品ページ時間',
        '直帰率', '離脱率', 'ページ価値', '特別な日', '週末', '新規']
print(len(cols), '本の列')


#### 1行ずつ読む

- 1行目：週末ならTrue。`astype(int)`で1と0に直し、列「週末」にしまう
- 2行目：初めての訪問（New_Visitor）なら1、そうでなければ0の列「新規」
- 3行目：購入したら1、しなければ0の列「購入」。これが答えの列
- 4〜6行目：手がかりに使う12本の列の名前を、リスト`cols`にしまう
- 7行目：リストの長さ（列の本数）を表示する


#### 中身を見る：0と1の列


In [ ]:
df[['Weekend', '週末', 'VisitorType', '新規']].head(3)


3件とも平日で、再訪（Returning_Visitor）。だから両方0


OS・ブラウザ・地域・流入元の列は、1、2、3…の番号で書いてあります。番号の大小には意味がない（地域3は地域1の3倍、ではない）ので、今日は使いません。月も今日は使いません。数の列10本と、0と1の列2本で進みます。


---


## 2. どんな訪問が買う？


読み込めました。次は、列が何を表すかと、答えの割合を見ます。そして、結果を見る前に、予想を書きます。


### 列の説明

| 列 | 中身 |
|---|---|
| 管理・情報・商品ページ数 | その種類のページを見た数（管理ページはアカウントなど、情報ページは配送やお知らせなど） |
| 管理・情報・商品ページ時間 | その種類のページにいた時間（秒） |
| 直帰率 | 最初のページだけ見て帰った人の割合（見たページの平均） |
| 離脱率 | そのページで訪問を終えた割合（見たページの平均） |
| ページ価値 | 見たページの「価値」の平均（4節で詳しく読む） |
| 特別な日 | 母の日などの特別な日に近いほど大きい（0〜1） |
| 週末・新規 | 週末の訪問か、初めての訪問か（0と1） |


### お手本：購入の割合

- 役割：買わなかった訪問（0）と購入（1）の件数と、購入の割合を出す
- 入力：表`df`の列「購入」
- 出力：件数と割合

0と1の列の平均は、1の割合です（第6回）。


In [ ]:
print(df['購入'].value_counts())
print(round(df['購入'].mean(), 3))


#### 1行ずつ読む

- 1行目：`value_counts()`で、0（買わない）と1（購入）の件数を数える
- 2行目：`mean()`で、1の割合を出し、小数第3位までに丸める


12,330件のうち、購入は1,908件です。購入の割合は0.155で、6〜7件に1件です。少ない側を見つけたい、という第6回と同じ形です。


### お手本：買った訪問と比べる

- 役割：買わなかった訪問と買った訪問で、商品ページにいた時間と離脱率の平均を並べる
- 入力：表`df`
- 出力：2行2列の表


In [ ]:
df.groupby('購入')[['商品ページ時間', '離脱率']].mean().round(3)


#### 1行ずつ読む

- `groupby('購入')`：列「購入」の値（0と1）で、行を2つの組に分ける
- `[['商品ページ時間', '離脱率']]`：2本の列を取り出す
- `mean().round(3)`：組ごとの平均を出し、小数第3位までに丸める


#### 中身を見る：組ごとの件数


In [ ]:
df.groupby('購入')['商品ページ時間'].count()


0の組が10,422件、1の組が1,908件。`count()`は組ごとの件数で、`value_counts()`と同じ数になる


商品ページにいた時間は、買わなかった訪問が約1,070秒、買った訪問が約1,876秒です。離脱率は、買わなかった訪問が0.047、買った訪問が0.020です。長くいて、途中で帰らない訪問ほど、買っています。


### 結果を見る前に、予想を書く

期末レポートでも、解析の前に予想を書きます。結果を見たあとでは書けません。予想は外れてかまいません。外れたところが、いちばんの発見になります。

12本の列のうち、購入を見分けるのにいちばん効くのは、どの列だと思いますか。理由も1行で書いてください（例：「いちばん効くのは、商品ページにいた時間だろう。長くいる人ほど買っていたから」）。


（ここに予想を書く）


### TODO①：データの番号を替える（予想 → 実行 → 比べる）

下のセルは、データの番号で表を取ってきて、特徴の表の形（行の数、列の数）と、答えの列の平均（1の割合）を表示します。

まず、予想を書いてください。番号を601（工作機械）にすると、特徴の列と答えの列は、それぞれ何本でしょうか。答えの列は何を表すでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`468`を`601`に替えて実行してください。


In [ ]:
# TODO: 468 を 601 に替えて実行する

uci_x = fetch_ucirepo(id=601)
print(uci_x.data.features.shape)
print(uci_x.data.targets.mean().round(3))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。列の本数と、答えの列の中身は、どうでしたか。


（ここに答えを書く）


工作機械は10,000件、列6本です。故障（Machine failure）は0.034で、ネットショップよりもっと偏っています。答えの列は6本あり、故障そのものと、故障の種類5つです。名前だけでは分からないことが、ページや表を見ると分かります。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 選ぶ・予想（1・2節） | UCIで3本を比べ、ネットショップの訪問を選んだ。購入は0.155 |
| 作って測る（3節） | ？（このあとの段で） |
| 確かめる（4節） | ？（このあとの段で） |
| しきい値を決める（5節） | ？（このあとの段で） |

まだ分からないのは、購入をどのくらい見分けられるか、です。


---


## 3. 購入を見分けられる？


ここから、段2「作って測る」（3節）です。問いは、12本の列から、購入をどのくらい見分けられるか、です。

ネットショップの訪問を選び、12本の列と答えの列「購入」を用意しました。ここから、確率を出して、混同行列とAUCで測ります（第6回の道具）。


### 何も言わなくても0.845

全部「買わない」と言えば、買わない訪問がすべて当たります。正解率は、1から購入の割合を引いた数です。


In [ ]:
print(round(1 - df['購入'].mean(), 3))


正解率だけでは分かりません。混同行列とAUCで測ります（第6回）。


### 単位がばらばら

商品ページ時間は秒で、いちばん長い訪問は6万秒を超えます。直帰率や離脱率は、0〜0.2の割合です。大きさがそろわないと、式の計算がうまく進まないことがあります。

そこで、第3回の標準化を使います。列$j$ごとに、平均$\bar{x}_j$を引き、標準偏差（standard deviation）$s_j$で割ります。

$$
z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}
$$

どの列も平均0、標準偏差1になります。平均と標準偏差は、訓練のデータだけで決め、テストにも訓練の値を使います。

`StandardScaler`は、`fit`で列ごとの平均と標準偏差を覚え、`transform`で標準化する道具です。`make_pipeline(A, B)`は、AとBをつなぎ、1つの式の作り方にする道具です。つないだ式の作り方を交差検証に渡すと、組ごとに、訓練の組だけで平均を覚えます。


### お手本：確率を出す

- 役割：標準化とロジスティック回帰をつなぎ、交差検証で、すべての訪問に購入の確率を付ける
- 入力：表`df`の12本の列`cols`と、列「購入」
- 出力：列「確率」と、確率のいちばん小さい値・大きい値


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict
model = make_pipeline(StandardScaler(), LogisticRegression())
p = cross_val_predict(model, df[cols], df['購入'], cv=5,
                      method='predict_proba')
df['確率'] = p[:, 1]
print(round(df['確率'].min(), 3), round(df['確率'].max(), 3))


#### 1行ずつ読む

- 1〜4行目：標準化、ロジスティック回帰、つなぐ道具、交差検証の道具を呼び出す
- 5行目：標準化とロジスティック回帰をつなぎ、式の作り方`model`にしまう
- 6・7行目：式の作り方、12本の列、答えの列を渡す。`cv=5`は5つの組
- `method='predict_proba'`で、どの訪問にも、自分を見ていない式から確率が付く（第6回）
- 8行目：`p[:, 1]`（すべての行の、1番の列＝購入の確率）を、列「確率」にしまう
- 9行目：確率のいちばん小さい値と、いちばん大きい値を表示する


#### make_pipelineがしていること

表を上から順に5つの組に分け、1つの組を残します。残りの4つの組で、列ごとの平均と標準偏差を覚え（`fit`）、標準化して式を作ります。残した組は、4つの組で覚えた平均と標準偏差で標準化し（`transform`）、確率を出します。これを5回くり返します。

| 回 | 平均・標準偏差を覚え、式を作る組 | 確率を出す組 |
|---:|---|---:|
| 1 | 2・3・4・5 | 1 |
| 2 | 1・3・4・5 | 2 |
| … | … | … |
| 5 | 1・2・3・4 | 5 |

5回とも、確率を出す組を見ないで、平均と標準偏差を覚えます。第3回で手で書いた「テストは訓練の平均と標準偏差で」を、道具が組ごとに行います。


#### 中身を見る：確率の列


In [ ]:
df[['購入', '確率']].head(3)


1件に1つずつ、購入の確率が付いた。先頭の3件は、どれも確率が小さい


### お手本：確率を描く

- 役割：買わなかった訪問と買った訪問で、確率の分かれ方を描く
- 入力：列「購入」と列「確率」
- 出力：2色の柱の図


In [ ]:
plt.hist(df[df['購入'] == 0]['確率'], bins=40, alpha=0.6,
         label='買わない')
plt.hist(df[df['購入'] == 1]['確率'], bins=40, alpha=0.6,
         label='買った')
plt.yscale('log')
plt.xlabel('購入の確率')
plt.ylabel('訪問の数（対数の目盛り）')
plt.legend()


#### 1行ずつ読む

- 1〜4行目：買わなかった訪問と買った訪問の確率を、40本の柱で重ねて描く
- 5行目：縦の目盛りを対数にする（0の近くの柱が高すぎて、ほかが見えないため）
- 6〜8行目：軸の名前と、色の説明（凡例）を付ける


買わなかった訪問は、ほとんどが0の近くに集まります。買った訪問は、0から1まで広く散らばります。0の近くにも、買った訪問がかなりあり、ここが見逃しになります。


### お手本：0.5で切る

- 役割：確率が0.5以上なら購入と判定し、混同行列を数える（第6回）
- 入力：列「確率」と列「購入」
- 出力：列「判定」と、2×2の表


In [ ]:
from sklearn.metrics import confusion_matrix
df['判定'] = (df['確率'] >= 0.5).astype(int)
print(confusion_matrix(df['購入'], df['判定']))


#### 1行ずつ読む

- 1行目：混同行列の道具を呼び出す
- 2行目：確率が0.5以上ならTrue。`astype(int)`で1と0に直し、列「判定」にしまう
- 3行目：本当（購入）と判定を渡し、2×2の表を表示する


#### 中身を見る：確率と判定


In [ ]:
df[['確率', '判定']].head(3)


確率が0.5より小さいので、3件とも判定は0（買わない）


In [ ]:
# @title 図：混同行列を2×2の表に描く（中身は読み飛ばしてよい）
vals = confusion_matrix(df['購入'], df['判定'])
plt.figure(figsize=(5, 4))
plt.imshow([[1, 0.5], [0, 1]], cmap='Pastel1', vmin=0, vmax=8)
names2 = [['買わないを当てた（TN）', '誤検出（FP）'], ['見逃し（FN）', '購入を当てた（TP）']]
for i in range(2):
    for j in range(2):
        plt.text(j, i - 0.15, names2[i][j], ha='center', fontsize=11)
        plt.text(j, i + 0.15, vals[i][j], ha='center', fontsize=18)
plt.xticks([0, 1], ['判定：買わない', '判定：購入'])
plt.yticks([0, 1], ['本当：買わない', '本当：購入'])
plt.title('0.5で切った混同行列')
plt.show()
print(vals)


買った1,908件のうち、1,217件を見逃しています。正解率は0.882で、何もしないとき（0.845）より少し高いだけです。


### お手本：AUCを出す

- 役割：しきい値を決めずに、確率の付き方そのものを測る（第6回）
- 入力：列「購入」と列「確率」
- 出力：AUC


In [ ]:
from sklearn.metrics import roc_auc_score
print(round(roc_auc_score(df['購入'], df['確率']), 3))


#### 1行ずつ読む

- 1行目：AUCの道具`roc_auc_score`を呼び出す
- 2行目：本当の答えと確率を渡すと、AUCが返る。しきい値は渡さない


AUCは0.877です。買った訪問1件と買わなかった訪問1件の組のうち、0.877の割合で、買った訪問の確率が高くなっています。


### TODO②：列を1本だけにする（予想 → 実行 → 比べる）

下のセルは、列の組`cols_x`で交差検証の確率を出し、AUCを表示します。

まず、予想を書いてください。商品ページにいた時間だけで確率を出すと、AUC（12本で0.877）はいくつになるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`cols`を`['商品ページ時間']`に替えて実行してください。


In [ ]:
# TODO: cols を ['商品ページ時間'] に替えて実行する

cols_x = ['商品ページ時間']
p = cross_val_predict(model, df[cols_x], df['購入'], cv=5,
                      method='predict_proba')
print(round(roc_auc_score(df['購入'], p[:, 1]), 3))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。1本だけのAUCは、12本と比べてどうでしたか。


（ここに答えを書く）


商品ページにいた時間だけでは、AUCは0.655です。12本の0.877より、ずっと低くなります。2節で差が見えた列でも、1本では見分けきれません。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 選ぶ・予想（1・2節） | UCIで3本を比べ、ネットショップの訪問を選んだ。購入は0.155 |
| 作って測る（3節） | 12本の列で確率。AUCは0.877。0.5では買った訪問の6割を見逃す |
| 確かめる（4節） | ？（このあとの段で） |
| しきい値を決める（5節） | ？（このあとの段で） |

まだ分からないのは、このAUCを信じてよいか、どの列が効いているか、です。


---


## 4. その成績、信じてよい？


ここから、段3「確かめる」（4節）です。問いは、AUC 0.877を信じてよいか、どの列が効いているか、です。

12本のうち、どの列が効いているのかを知るために、1本ずつ外して、AUCが下がるかを見ます。列を1本外して、確率を出し、AUCを測る。これを12本ぶん行います。同じ3行を12回書くのは、まちがいのもとです。そこで、手順に名前を付けて、関数にまとめます。


### 関数：第3回・第6回のおさらい

`def 名前(入力):`の下の字下げした行が、関数の中身で、呼ぶたびに動きます。`return`で答えを返します。入力を替えて呼ぶと、同じ手順が別の入力で動きます。


In [ ]:
def double(x):
    return 2 * x

print(double(3))
print(double(10))


### お手本：手順を関数にまとめる

- 役割：列の組を受け取り、交差検証で確率を出して、AUCを返す
- 入力：列の名前のリスト`cols`
- 出力：AUC（小数第3位まで）


In [ ]:
def auc_of(cols):
    p = cross_val_predict(model, df[cols], df['購入'], cv=5,
                          method='predict_proba')
    return round(roc_auc_score(df['購入'], p[:, 1]), 3)

print(auc_of(cols))


#### 1行ずつ読む

- 1行目：関数`auc_of`を作る。入力は、列の名前のリスト`cols`
- 2・3行目：3節と同じ。渡された列で、交差検証の確率を出す
- 4行目：AUCを出し、小数第3位までに丸めて返す
- 6行目：12本の列で呼ぶ。3節と同じ0.877が返る


#### 中身を見る：入力を替えて呼ぶ


In [ ]:
print(auc_of(['商品ページ時間']))
print(auc_of(['ページ価値']))


1行で、別の列の組を試せる。ページ価値1本だけで、0.830もある


### 関数は、手順に付けた名前

下の図は、関数`auc_of`に、3つの入力を渡したときの様子です。手順は1つで、入力（列の組）を替えて呼ぶたびに、同じ手順が動きます。


In [ ]:
# @title 図：関数に入力を替えて渡す（中身は読み飛ばしてよい）
for ins in [['商品ページ時間'], ['ページ価値'], cols]:
    print(len(ins), '本の列 → auc_of →', auc_of(ins))


> 今日の重要概念（Python）：関数
>
> 関数は、手順に名前を付けたもの。入力を替えて呼ぶたびに、同じ手順が動き、答えを返す。


### お手本：1本ずつ外す

- 役割：12本の列から1本ずつ外し、残りの11本でAUCを測る
- 入力：リスト`cols`と関数`auc_of`
- 出力：外した列の名前と、そのときのAUC（12行）


In [ ]:
for c in cols:
    rest = []
    for x in cols:
        if x != c:
            rest.append(x)
    print(c, auc_of(rest))


#### 1行ずつ読む

- 1行目：12本の列の名前を1つずつ`c`に入れ、字下げした行をくり返す。`c`が外す列
- 2行目：残す列をためる、空のリスト`rest`を作る
- 3行目：もう一度、12本の列を1つずつ`x`に入れてくり返す
- 4・5行目：`x`が外す列`c`でなければ（`!=`）、`rest`の後ろに足す
- 6行目：外した列の名前と、残りの11本で測ったAUCを表示する
- 関数があるので、くり返しの中身が1行で済む


#### 中身を見る：残した列


In [ ]:
c = 'ページ価値'
rest2 = []
for x in cols:
    if x != c:
        rest2.append(x)
print(len(rest2), 'ページ価値' in rest2)


外す列をページ価値に決めて、1回だけ回した。11本が残り、ページ価値は入っていない


えっ、ほかの列は、外してもAUCはほとんど変わりません（0.875〜0.882）。ページ価値を外したときだけ、0.877から0.718に下がります。

2節の予想と比べてみてください。たとえば「いちばん効くのは、商品ページにいた時間」と予想していたら、外れです。ページ価値1本で、ほとんど決まっていました。ページ価値とは、何なのでしょうか。


### お手本：ページ価値がある訪問

- 役割：ページ価値が0より大きい訪問の割合を、買わなかった訪問と買った訪問で比べる
- 入力：表`df`
- 出力：組ごとの割合


In [ ]:
df['価値あり'] = (df['ページ価値'] > 0).astype(int)
print(df.groupby('購入')['価値あり'].mean().round(3))


#### 1行ずつ読む

- 1行目：ページ価値が0より大きければ1、0なら0の列「価値あり」を作る
- 2行目：購入の0と1で組に分け、「価値あり」の平均（1の割合）を出す


#### 中身を見る：ページ価値と価値あり


In [ ]:
df[['ページ価値', '価値あり', '購入']].head(3)


先頭の3件は、どれもページ価値が0なので、価値ありも0。購入も0


ページ価値が0より大きい訪問は、買わなかった訪問では0.114、買った訪問では0.806です。買った訪問の8割に、ページ価値があります。


### データの説明を読む

UCIのページには、こう書いてあります。

> The "Page Value" feature represents the average value for a web page that a user visited before completing an e-commerce transaction.

訳：ページ価値は、購入を終える前に見たページの、価値の平均。

ページの価値は、アクセス解析の道具（Google Analytics）が出す数です。購入の売上を、その購入の前に見られたページに割り振って、ページごとに計算します。購入が無ければ、ページの価値は付きません。答え（購入）から作られた列です。

では、使ってよいのでしょうか。過去の購入から前もって計算しておいた値なら、訪問の途中でも分かるので、使ってかまいません。この訪問の購入まで入れて計算した値なら、答えを見ていることになります。このデータの説明には、いつの購入から計算したかは書いてありません。


### お手本：ページ価値なしで切る

- 役割：ページ価値を除いた11本で確率を出し、0.5で切った混同行列を数える
- 入力：リスト`cols`
- 出力：2×2の表


In [ ]:
no_pv = []
for x in cols:
    if x != 'ページ価値':
        no_pv.append(x)
p = cross_val_predict(model, df[no_pv], df['購入'], cv=5,
                      method='predict_proba')
judge = (p[:, 1] >= 0.5).astype(int)
print(confusion_matrix(df['購入'], judge))
print(judge.sum(), round((judge == df['購入']).mean(), 3))


#### 1行ずつ読む

- 1〜4行目：ページ価値を除いた11本の列を、リスト`no_pv`にためる
- 5・6行目：11本で、交差検証の確率を出す
- 7行目：0.5で切った判定を`judge`にしまう。8行目：混同行列を表示する
- 9行目：購入と判定した件数と、判定が当たった割合（正解率）を表示する


#### 中身を見る：11本の列


In [ ]:
print(len(no_pv))
print(no_pv[-4:])


11本。後ろの4本に、ページ価値は無い（離脱率の次が特別な日）


ページ価値なしでは、購入と判定したのは30件だけです。正解率0.844は、何も言わないとき（0.845）より低くなります。

どちらで進むかを決めます。ここでは、店が過去の購入からページ価値を前もって計算している、と仮定します。この仮定のもとで、ページ価値を入れた12本で進みます。そして、仮定が崩れたら、AUCは0.718まで下がる、と書き添えます。期末レポートの「確かめた」は、この形で書きます。


> 重要：成績が良すぎる列は、データの説明を読み、何から作られた値かを確かめる。答えから作られた列なら、使ってよいかは、いつ計算したかで決まる。


### TODO③：外す列を替える（予想 → 実行 → 比べる）

下のセルは、列`c`を外した残りの列でAUCを測ります。

まず、予想を書いてください。離脱率を外すと、AUC（12本で0.877）はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`'ページ価値'`を`'離脱率'`に替えて実行してください。


In [ ]:
# TODO: 'ページ価値' を '離脱率' に替えて実行する

c = '離脱率'
rest_x = []
for x in cols:
    if x != c:
        rest_x.append(x)
print(auc_of(rest_x))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。離脱率を外したAUCは、ページ価値を外したときと比べてどうでしたか。


（ここに答えを書く）


離脱率を外すと0.882です。下がるどころか、わずかに上がりました。外しても困らない列もあります。効いているのは、ページ価値でした。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 選ぶ・予想（1・2節） | UCIで3本を比べ、ネットショップの訪問を選んだ。購入は0.155 |
| 作って測る（3節） | 12本の列で確率。AUCは0.877。0.5では買った訪問の6割を見逃す |
| 確かめる（4節） | ページ価値を外すとAUC 0.718。購入から計算された列。仮定を置いて使う |
| しきい値を決める（5節） | ？（このあとの段で） |

まだ分からないのは、確率をどこで切ればよいか、です。


---


## 5. どこで切る？


ここから、段4「しきい値を決める」（5節）です。問いは、確率をどこで切って、声をかける訪問を決めるか、です。

12本の列で確率を出し、AUCは0.877でした。ページ価値は、仮定を置いて使います。0.5で切ると、買った1,908件のうち1,217件を見逃しました。0.5でよいのでしょうか。2つの誤りの重さから決めます。


### 2つの誤りに、重さを置く

確率がしきい値以上の訪問に、チャットの担当者が声をかけます。見逃しは、買う気のある訪問に声をかけそびれることで、売上の機会を逃します。誤検出は、買わない訪問に担当者の時間を使うことです。

誤検出1件の損を1とし、見逃し1件の損を$c_{FN}$とします。

$$
\text{損の合計} = c_{FN} \times \text{見逃しの件数} + 1 \times \text{誤検出の件数}
$$

ここでは、見逃しは誤検出の5倍重い（$c_{FN} = 5$）と置きます。これは仮定です。0.5で切ると、見逃し1,217件、誤検出239件なので、損の合計は5 × 1,217 + 239 = 6,324です。

しきい値を下げると、見逃しが減り、誤検出が増えます（第6回）。損はどうなるでしょうか。


### お手本：しきい値ごとの損

- 役割：しきい値を0.01から0.99まで動かし、そのたびに損の合計を数えて、リストにためる
- 入力：列「確率」と列「購入」、見逃しの重さ`c_fn`
- 出力：リスト`ts`（しきい値）と`costs`（損の合計）


In [ ]:
c_fn = 5
ts = []
costs = []
for k in range(1, 100):
    t = k / 100
    m = confusion_matrix(df['購入'], (df['確率'] >= t).astype(int))
    ts.append(t)
    costs.append(c_fn * m[1, 0] + m[0, 1])
print(ts[0], ts[-1], costs[49])


#### 1行ずつ読む

- 1行目：見逃し1件の重さ`c_fn`を5にする
- 2・3行目：しきい値をためるリスト`ts`と、損をためるリスト`costs`を、空で作る
- 4行目：`k`を1から99まで1つずつ変えて、くり返す
- 5行目：`k / 100`で、しきい値`t`は0.01から0.99まで
- 6行目：しきい値`t`で切った判定の混同行列を、`m`にしまう
- 7行目：しきい値を`ts`に足す
- 8行目：`m[1, 0]`は見逃し、`m[0, 1]`は誤検出（第6回）。損の合計を`costs`に足す
- 9行目：最初と最後のしきい値と、50番目（しきい値0.5）の損を表示する


#### 中身を見る：最初の3回


In [ ]:
for k in range(1, 4):
    t = k / 100
    m = confusion_matrix(df['購入'], (df['確率'] >= t).astype(int))
    print(t, m[1, 0], m[0, 1], c_fn * m[1, 0] + m[0, 1])


しきい値がとても低いと、見逃しは少ないが、誤検出が数千件。損は大きい


### お手本：損を描く

- 役割：しきい値と損の合計を線で描く
- 入力：リスト`ts`と`costs`
- 出力：線の図


In [ ]:
plt.plot(ts, costs)
plt.xlabel('しきい値')
plt.ylabel('損の合計')


#### 1行ずつ読む

- 1行目：横にしきい値、縦に損の合計を取り、99個の点を線でつなぐ
- 2・3行目：軸の名前を付ける


損には谷があります。しきい値が低すぎると、誤検出で損が増えます。高すぎると、見逃しで損が増えます。0.5は、谷の底から遠いところです。谷の底のしきい値で切れば、損がいちばん小さくなります。


### お手本：いちばん小さいところ

- 役割：損の合計がいちばん小さいしきい値を探す
- 入力：リスト`ts`と`costs`
- 出力：しきい値と、そのときの損


In [ ]:
i = costs.index(min(costs))
print(ts[i], costs[i])


#### 1行ずつ読む

- 1行目：`min(costs)`で、いちばん小さい損を出す
- 続けて`costs.index(…)`で、それがリストの何番目かを調べ、`i`にしまう
- 2行目：`i`番目のしきい値と、そのときの損を表示する


#### 中身を見る：indexの動き


In [ ]:
a = [30, 10, 20]
print(min(a), a.index(min(a)))


いちばん小さいのは10で、リストの1番目（0から数える）


In [ ]:
# @title 図：混同行列を2×2の表に描く（中身は読み飛ばしてよい）
vals = confusion_matrix(df['購入'], (df['確率'] >= ts[i]).astype(int))
plt.figure(figsize=(5, 4))
plt.imshow([[1, 0.5], [0, 1]], cmap='Pastel1', vmin=0, vmax=8)
names2 = [['買わないを当てた（TN）', '誤検出（FP）'], ['見逃し（FN）', '購入を当てた（TP）']]
for i in range(2):
    for j in range(2):
        plt.text(j, i - 0.15, names2[i][j], ha='center', fontsize=11)
        plt.text(j, i + 0.15, vals[i][j], ha='center', fontsize=18)
plt.xticks([0, 1], ['判定：買わない', '判定：購入'])
plt.yticks([0, 1], ['本当：買わない', '本当：購入'])
plt.title('0.15で切った混同行列')
plt.show()
print(vals)


見逃しを5倍と置くと、0.15で切るのが、損がいちばん小さくなります。見逃しは1,217件から481件に減り、誤検出は239件から1,344件に増えました。損の合計は、6,324から3,749に下がります。


> 今日の重要概念（統計）：しきい値は誤りの重さで決める
>
> しきい値は0.5に決まっていない。見逃しと誤検出に重さを置き、損の合計がいちばん小さいところで切る。


In [ ]:
# @title 表：重さを替えると（中身は読み飛ばしてよい）
for w in [1, 2, 5, 10]:
    cs = []
    for t in ts:
        m = confusion_matrix(df['購入'], (df['確率'] >= t).astype(int))
        cs.append(w * m[1, 0] + m[0, 1])
    j = cs.index(min(cs))
    m = confusion_matrix(df['購入'], (df['確率'] >= ts[j]).astype(int))
    print('重さ', w, ' しきい値', ts[j], ' 見逃し', m[1, 0], ' 誤検出', m[0, 1])


### 重さを替えると

| 見逃しの重さ $c_{FN}$ | しきい値 | 見逃し | 誤検出 |
|---:|---:|---:|---:|
| 1 | 0.36 | 1,008 | 368 |
| 2 | 0.18 | 595 | 902 |
| 5 | 0.15 | 481 | 1,344 |
| 10 | 0.12 | 335 | 2,286 |

見逃しを重くするほど、しきい値は下がり、たくさんの訪問に声をかけます。

重さは、データからは出てきません。見逃し1件と誤検出1件の重さは、店が決めます。1件の売上、担当者の数と時間、お客の印象などで決まります。期末レポートでは、重さを仮定として書き、替えたらどうなるかも見せます。


### TODO④：見逃しの重さを替える（予想 → 実行 → 比べる）

下のセルは、見逃しの重さ`c_fn`で、しきい値ごとの損を数え、いちばん小さいしきい値と損を表示します。

まず、予想を書いてください。見逃しと誤検出を同じ重さ（1）にすると、しきい値（5倍で0.15）はどうなるでしょうか。0.5になるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`5`を`1`に替えて実行してください。


In [ ]:
# TODO: 5 を 1 に替えて実行する

c_fn = 1
costs_x = []
for t in ts:
    m = confusion_matrix(df['購入'], (df['確率'] >= t).astype(int))
    costs_x.append(c_fn * m[1, 0] + m[0, 1])
i = costs_x.index(min(costs_x))
print(ts[i], costs_x[i])


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。同じ重さのときのしきい値は、0.5でしたか。


（ここに答えを書く）


同じ重さなら、しきい値は0.36です。0.5ではありません。ロジスティック回帰の確率は、ぴったり正しい確率とは限りません。だから、式で決めずに、損を数えて決めます。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 選ぶ・予想（1・2節） | UCIで3本を比べ、ネットショップの訪問を選んだ。購入は0.155 |
| 作って測る（3節） | 12本の列で確率。AUCは0.877。0.5では買った訪問の6割を見逃す |
| 確かめる（4節） | ページ価値を外すとAUC 0.718。購入から計算された列。仮定を置いて使う |
| しきい値を決める（5節） | 見逃しを5倍と置くと、0.15で切る。損は6,324から3,749に |

4つの段がそろい、今日の問いに答えられます。


### 答え

12本の列でロジスティック回帰の確率を出すと、AUCは0.877です。効いているのはページ価値で、購入から計算された列なので、仮定を置いて使います。見逃しを誤検出の5倍と置くと、0.15で切るのが、損の合計がいちばん小さくなります。そのとき、買った1,908件のうち1,427件に声をかけ、買わない1,344件にも声をかけます。


まだ言えないこと

- ページ価値の仮定：いつの購入から計算したかは、店に聞かないと分からない
- 重さ：5倍という数は仮定。実際の売上と担当者の時間から決めたい
- 声をかけたら買うか：このデータは、声をかけなかったときの記録


### レポートなら、こう書く

今日の解析を、期末レポートの項目に写すと、次のようになります。

- 選んだ理由：実際のネットショップの記録で、1行が身近。作ったデータの2本は外した
- 予想：いちばん効くのは、商品ページにいた時間だろう
- 解析：12本の列でロジスティック回帰の確率を出し、AUCは0.877
- 確かめた：1本ずつ外すと、ページ価値だけで0.718に下がる。説明を読み、仮定を置いた
- 驚き：予想した商品ページの時間ではなく、ページ価値1本でほぼ決まっていた
- 決めた：見逃しを5倍と置き、0.15で切る。重さを替えた表も付ける


---


## 目標に答えられたか


- 今日の目標は「ネットショップの訪問から、購入を見分けられるか。見分けたとして、どこで切ればよいか」でした。AUCと、しきい値と、損の合計で答えられますか？
- 1節で、工作機械ときのこのデータを外したのはなぜですか？
- 3節で、`make_pipeline`でつないだ標準化は、どの組のデータで平均と標準偏差を覚えましたか？
- 4節で、関数`auc_of`を作ると、何が楽になりましたか？
- 4節で、ページ価値を使ってよいかが、データの説明だけでは決まらないのはなぜですか？
- 5節で、見逃しを重くすると、しきい値はどちらへ動きましたか？


---


## 今日の要点


- 今日、持ち帰る1文：しきい値は0.5に決まっていない。見逃しと誤検出の重さを置けば、しきい値が決まる。そして、成績を決めている列が何から作られたかを、データの説明で確かめる
- 重要概念（統計）：しきい値は誤りの重さで決める。見逃しと誤検出に重さを置き、損の合計がいちばん小さいところで切る
- 重要概念（Python）：関数。手順に名前を付けたもの。入力を替えて呼ぶたびに、同じ手順が動き、答えを返す
- データは、ページの説明まで読んで選ぶ。作ったデータか、人を判定しないか
- 成績が良すぎる列は、何から作られたかを確かめる。重さは仮定として書く


---


## 次回へ


今日は、答えの列（購入）があるデータで、確率を出して切りました。では、答えの列が無いとき、似た行をまとめて、群に分けられるでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：しきい値は誤りの重さで決める。しきい値は0.5に決まっていない。見逃しと誤検出に重さを置き、損の合計がいちばん小さいところで切る（5節）
- Python：関数。手順に名前を付けたもの。入力を替えて呼ぶたびに、同じ手順が動き、答えを返す（4節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

しきい値と誤りの重さなら、たとえば「天気予報を見て、傘を持っていくか」を描けます。
濡れるのがとても嫌な人は、降水確率20%でも持っていきます。荷物が重いのが嫌な人は、70%まで持ちません。同じ予報でも、誤りの重さで決め方が変わります。5節の損の谷の図が手本になります。

</details>


<details><summary>詰まったら</summary>

関数なら、たとえば「炊飯器の『炊く』ボタン」を描けます。
手順（水を吸わせる、加熱する、蒸らす）は1つです。お米を替えて押すたびに、同じ手順が動き、ごはんが返ります。4節の`auc_of`に3つの入力を渡す図が手本になります。

</details>


---


## 発展（任意）


### 月も、手がかりにしたら？

購入の割合は、月で違います（11月は0.254、2月は0.016）。月を0と1の列にして足すと、AUCは上がるでしょうか。`pd.get_dummies`は、月ごとに「その月なら1」の列を作る道具です。


In [ ]:
print(df.groupby('Month')['購入'].mean().round(3))


In [ ]:
mon = pd.get_dummies(df['Month'], dtype=int)
X_mon = pd.concat([df[cols], mon], axis=1)
p = cross_val_predict(model, X_mon, df['購入'], cv=5,
                      method='predict_proba')
print(round(roc_auc_score(df['購入'], p[:, 1]), 3))


#### 1行ずつ読む

- 1行目：月ごとの0と1の列（10か月ぶん）を作る
- 2行目：12本の列と月の列を、横につなぐ
- 3〜5行目：交差検証の確率を出し、AUCを表示する


月を足したら、0.877から0.859に下がりました。下の表で、組ごとの月の件数を見ます。


In [ ]:
fold = np.arange(len(df)) * 5 // len(df) + 1
pd.crosstab(fold, df['Month'])[['Feb', 'Mar', 'May', 'Nov', 'Dec']]


#### 1行ずつ読む

- 1行目：上から順に5つの組に分けたときの、組の番号（1〜5）を作る
- 2行目：組と月の組み合わせごとに件数を数え（`pd.crosstab`）、5つの月を表示する


表はほぼ月の順に並んでいて、上から5つに分けると、2月と3月は1つ目の組にしかありません。1つ目の組の確率を出すとき、式は2月と3月を一度も見ていないのです。

行を混ぜてから5つの組に分けると、どうなるでしょうか。`KFold(5, shuffle=True, random_state=0)`は、行を混ぜてから5つに分ける分け方です。


In [ ]:
from sklearn.model_selection import KFold
shuf = KFold(5, shuffle=True, random_state=0)
p = cross_val_predict(model, X_mon, df['購入'], cv=shuf,
                      method='predict_proba')
print(round(roc_auc_score(df['購入'], p[:, 1]), 3))


行を混ぜると、月を足して0.894に上がります。どちらが正しいかは、何を当てたいかで決まります。来月の訪問を当てたいなら、上から順に分けたほうが、実際の使い方に近い分け方です。

試すなら、月の代わりに、流入元（`TrafficType`）を`pd.get_dummies`で足してみましょう。番号の列も、0と1の列にすれば使えます。
